In [1]:
import pandas as pd
import numpy as np
import joblib
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

In [2]:
df = pd.read_csv("../data/telco_clean.csv")
df.shape

(7032, 21)

In [3]:
df = df.drop("customerID", axis=1)
df.shape

(7032, 20)

In [4]:
df["Churn"] = df["Churn"].map({"No": 0, "Yes": 1})
df["Churn"].value_counts()

Churn
0    5163
1    1869
Name: count, dtype: int64

In [5]:
df = pd.get_dummies(df, drop_first=True, dtype=int)
df.shape

(7032, 31)

In [6]:
df.head()

,SeniorCitizen,tenure,MonthlyCharges,TotalCharges,Churn,gender_Male,Partner_Yes,Dependents_Yes,PhoneService_Yes,MultipleLines_No phone service,...,StreamingTV_No internet service,StreamingTV_Yes,StreamingMovies_No internet service,StreamingMovies_Yes,Contract_One year,Contract_Two year,PaperlessBilling_Yes,PaymentMethod_Credit card (automatic),PaymentMethod_Electronic check,PaymentMethod_Mailed check
0,0,1,29.85,29.85,0,0,1,0,0,1,...,0,0,0,0,0,0,1,0,1,0
1,0,34,56.95,1889.50,0,1,0,0,1,0,...,0,0,0,0,1,0,0,0,0,1
2,0,2,53.85,108.15,1,1,0,0,1,0,...,0,0,0,0,0,0,1,0,0,1
3,0,45,42.30,1840.75,0,1,0,0,0,1,...,0,0,0,0,1,0,0,0,0,0
4,0,2,70.70,151.65,1,0,0,0,1,0,...,0,0,0,0,0,0,1,0,1,0


In [7]:
X = df.drop("Churn", axis=1)
y = df["Churn"]
print(X.shape)
print(y.shape)

(7032, 30)
(7032,)


In [8]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print(X_train.shape)
print(X_test.shape)

(5625, 30)
(1407, 30)


In [9]:
print(y_train.mean())
print(y_test.mean())

0.2657777777777778
0.2658137882018479


In [10]:
num_cols = ["tenure", "MonthlyCharges", "TotalCharges"]

scaler = StandardScaler()
X_train[num_cols] = scaler.fit_transform(X_train[num_cols])
X_test[num_cols] = scaler.transform(X_test[num_cols])

X_train[num_cols].head()

,tenure,MonthlyCharges,TotalCharges
1408,1.321816,0.981556,1.659900
6992,-0.267410,-0.971546,-0.562252
3349,1.444064,0.837066,1.756104
4486,-1.204646,0.641092,-0.908326
3535,0.669826,-0.808787,-0.101561


In [11]:
joblib.dump(scaler, "../models/scaler.pkl")
joblib.dump(list(X.columns), "../models/columns.pkl")

['../models/columns.pkl']

In [12]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score

In [13]:
print(X_train.shape, X_test.shape)

(5625, 30) (1407, 30)


In [14]:
n_stay = (y_train == 0).sum()
n_churn = (y_train == 1).sum()
ratio = n_stay / n_churn
print(n_stay, n_churn, round(ratio, 2))

4130 1495 2.76


In [15]:
lr = LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42)
lr.fit(X_train, y_train)
print("Logistic Regression trained")

Logistic Regression trained


In [16]:
rf = RandomForestClassifier(
    n_estimators=200, max_depth=8,
    class_weight="balanced", random_state=42
)
rf.fit(X_train, y_train)
print("Random Forest trained")

Random Forest trained


In [17]:
xgb = XGBClassifier(
    n_estimators=200, learning_rate=0.05, max_depth=3,
    scale_pos_weight=ratio, random_state=42,
    eval_metric="logloss"
)
xgb.fit(X_train, y_train)
print("XGBoost trained")

XGBoost trained


In [18]:
models = {"Logistic Regression": lr, "Random Forest": rf, "XGBoost": xgb}

for name, model in models.items():
    train_acc = accuracy_score(y_train, model.predict(X_train))
    test_acc = accuracy_score(y_test, model.predict(X_test))
    print(f"{name}: train = {train_acc:.3f}, test = {test_acc:.3f}")

Logistic Regression: train = 0.756, test = 0.726
Random Forest: train = 0.795, test = 0.738
XGBoost: train = 0.768, test = 0.730


In [19]:
sample = X_test.iloc[[0]]
print("Actual:", y_test.iloc[0])
print("Predicted:", xgb.predict(sample)[0])
print("Churn probability:", round(xgb.predict_proba(sample)[0][1], 3))

Actual: 0
Predicted: 0
Churn probability: 0.034


In [20]:
churners = y_test[y_test == 1].index[0]
sample2 = X_test.loc[[churners]]
print("Actual:", y_test.loc[churners])
print("Predicted:", xgb.predict(sample2)[0])
print("Churn probability:", round(xgb.predict_proba(sample2)[0][1], 3))

Actual: 1
Predicted: 0
Churn probability: 0.288


In [21]:
joblib.dump(lr, "../models/logistic_model.pkl")
joblib.dump(rf, "../models/rf_model.pkl")
joblib.dump(xgb, "../models/xgb_model.pkl")

['../models/xgb_model.pkl']